# ANN Implementation — Customer Churn Prediction
Predicts whether a bank customer will exit based on demographic and account features.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shrutimechlearn/churn-modelling")
print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

## 1. Load Data

In [ ]:
df = pd.read_csv(os.path.join(path, 'Churn_Modelling.csv'))
df.head()

In [ ]:
df.shape

## 2. Data Preprocessing
Goal: predict whether the customer will exit (`Exited` column).

In [ ]:
# Select input and output features
X = df.iloc[:, 3:-1].copy()
Y = df.iloc[:, -1]

In [ ]:
# One-hot encode categorical columns
geography = pd.get_dummies(X['Geography'], drop_first=True, dtype=int)
gender    = pd.get_dummies(X['Gender'],    drop_first=True, dtype=int)

X.drop(['Geography', 'Gender'], axis=1, inplace=True)
X = pd.concat([X, geography, gender], axis=1)

print("Feature matrix shape:", X.shape)

In [ ]:
# Train / test split
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=0
)

In [ ]:
# Feature scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test  = sc.transform(X_test)

print("X_train shape:", X_train.shape)

## 3. Build and Train ANN

In [ ]:
n_features = X_train.shape[1]

classifier = Sequential([
    Input(shape=(n_features,)),          # FIX: explicit input shape
    Dense(units=11, activation='relu'),  # Input layer
    Dense(units=7,  activation='relu'),  # Hidden layer 1
    Dropout(0.3),                        # FIX: Dropout added separately
    Dense(units=6,  activation='relu'),  # Hidden layer 2
    Dense(units=1,  activation='sigmoid')  # Output layer
])

classifier.summary()

In [ ]:
# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    min_delta=0.0001,
    patience=20,
    verbose=1,
    mode='auto',
    restore_best_weights=True
)

In [ ]:
classifier.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model_history = classifier.fit(
    X_train, Y_train,
    validation_split=0.33,
    batch_size=10,
    epochs=1000,
    callbacks=[early_stopping]
)

## 4. Evaluate Base Model

In [ ]:
# Plot training history
plt.plot(model_history.history['accuracy'])
plt.plot(model_history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# Predict on test set
Y_pred = classifier.predict(X_test)
Y_pred = (Y_pred > 0.5)
from sklearn.metrics import classification_report

print("\nClassification Report:")
print(classification_report(Y_test, Y_pred))

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(Y_test, Y_pred)
print("Confusion Matrix:\n", cm)

In [ ]:
# FIX: correct argument order — (y_true, y_pred)
score = accuracy_score(Y_test, Y_pred)
print(f"Accuracy: {score:.4f}")

## 5. Hyperparameter Tuning with Keras Tuner

In [ ]:
# pip install keras-tuner --upgrade

In [ ]:
# FIX: all imports at the top, before build_model is defined
import keras_tuner as kt
from keras_tuner.tuners import RandomSearch

In [ ]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(n_features,)))  # FIX: explicit input shape

    for i in range(hp.Int('layers', min_value=2, max_value=20, step=1)):
        model.add(Dense(
            units=hp.Int(f'units_{i}', min_value=32, max_value=512, step=32),
            activation=hp.Choice(f'activation_{i}', ['relu', 'tanh', 'sigmoid'])
        ))
        model.add(Dropout(
            rate=hp.Float(f'dropout_rate_{i}', min_value=0.0, max_value=0.5, step=0.1)
        ))

    model.add(Dense(1, activation='sigmoid'))

    optimizer_name = hp.Choice('optimizer_type', ['Adam', 'Adagrad', 'RMSprop'])
    learning_rate  = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])

    # FIX: keras is now imported at the top
    if optimizer_name == 'Adam':
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'Adagrad':
        optimizer = keras.optimizers.Adagrad(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=3,
    executions_per_trial=3,
    directory='project',
    project_name='Churn'
)

tuner.search_space_summary()

In [ ]:
tuner.search(
    X_train, Y_train,
    epochs=5,
    validation_data=(X_test, Y_test),
    callbacks=[early_stopping]
)

In [ ]:
print("Best hyperparameters:", tuner.get_best_hyperparameters()[0].values)

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
# FIX: store fit result so we can plot history
tuned_history = best_model.fit(
    X_train, Y_train,
    epochs=100,
    validation_data=(X_test, Y_test),
    callbacks=[early_stopping]
)

In [ ]:
# FIX: use tuned_history.history, not model.history
plt.plot(tuned_history.history['accuracy'])
plt.plot(tuned_history.history['val_accuracy'])
plt.title('Tuned Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# Final evaluation on test data
Y_pred_tuned = best_model.predict(X_test)
Y_pred_tuned = (Y_pred_tuned > 0.5)

# FIX: correct argument order
tuned_score = accuracy_score(Y_test, Y_pred_tuned)
print(f"Tuned Model Accuracy: {tuned_score:.4f}")